# 39 — IBM Hardware Tests v2: Fair DLA Parity + Sector Decoherence + FIM Protection

Previous IBM run (NB14) had circuit depth artefact: even=3 depth, odd=66.
This notebook designs THREE fair experiments based on our synthesis.

## Budget: ~8 min QPU remaining on ibm_fez

### Experiment A: Equal-Depth DLA Parity (fixes previous artefact)
Both parity states prepared via SAME fixed-depth ansatz (HEA, 4 layers).
Parameters optimised via VQE to approximate even/odd ground states.
Equal depth → any fidelity difference is INTRINSIC to parity.

### Experiment B: Magnetisation Sector Decoherence (tests NB38 mechanism)
Prepare states in M=0, M=±2, M=±4 sectors (n=4 qubits).
Apply identical noise layers. Measure sector-resolved fidelity.
NB38 prediction: sectors are energetically separated → different decoherence.

### Experiment C: FIM vs Standard Ground State (tests dual protection)
Prepare ground states of H_XY and H_FIM (with λ=2).
Apply identical noise. If H_FIM ground state is more robust → dual protection.

All circuits designed for equal depth within each experiment.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
from scipy.optimize import minimize

sys.path.insert(
    0,
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/src",
)
from scpn_quantum_control.bridge.knm_hamiltonian import (
    OMEGA_N_16,
    build_knm_paper27,
    knm_to_hamiltonian,
)

N = 4
K = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

# Build both Hamiltonians
H_xy_op = knm_to_hamiltonian(K, omega)
H_xy = H_xy_op.to_matrix()

# FIM Hamiltonian: H_FIM = H_XY - λ·M²/n
dim = 2**N
H_fim_correction = np.zeros((dim, dim))
for state in range(dim):
    m = sum(1 - 2 * ((state >> (N - 1 - i)) & 1) for i in range(N))
    H_fim_correction[state, state] = -2.0 * m**2 / N  # λ=2

H_fim = H_xy + H_fim_correction

# Diagonalise both
E_xy, V_xy = np.linalg.eigh(H_xy)
E_fim, V_fim = np.linalg.eigh(H_fim)

# Parity operator
P = SparsePauliOp("Z" * N).to_matrix()

print(f"H_XY ground energy: {E_xy[0]:.4f}")
print(f"H_FIM ground energy: {E_fim[0]:.4f}")
print(f"H_XY ground parity: {np.real(V_xy[:, 0].conj() @ P @ V_xy[:, 0]):+.4f}")
print(f"H_FIM ground parity: {np.real(V_fim[:, 0].conj() @ P @ V_fim[:, 0]):+.4f}")

## Experiment A: Equal-Depth DLA Parity

Use EfficientSU2 ansatz with FIXED reps=3 for BOTH even and odd states.
VQE optimises parameters to approximate each target.

In [ ]:
# Find even and odd ground states
sv_even = sv_odd = None
for i in range(dim):
    v = V_xy[:, i]
    parity = float(np.real(v.conj() @ P @ v))
    if parity > 0.5 and sv_even is None:
        sv_even = v
        print(f"Even: E_{i} = {E_xy[i]:.4f}")
    elif parity < -0.5 and sv_odd is None:
        sv_odd = v
        print(f"Odd:  E_{i} = {E_xy[i]:.4f}")
    if sv_even is not None and sv_odd is not None:
        break

# Fixed-depth ansatz — decompose to basic gates for Aer compatibility
REPS = 3
ansatz_raw = EfficientSU2(N, reps=REPS, entanglement="linear")
# Decompose to basic gates
from qiskit import transpile

ansatz = transpile(ansatz_raw, basis_gates=["cx", "rz", "sx", "x", "id"], optimization_level=0)
n_params = ansatz.num_parameters
print(
    f"Ansatz: EfficientSU2 decomposed, reps={REPS}, {n_params} parameters, depth={ansatz.depth()}"
)


def vqe_cost(params, target_sv):
    """Negative fidelity with target state."""
    bound = ansatz.assign_parameters(params)
    sv = Statevector.from_instruction(bound)
    fid = np.abs(np.dot(sv.data.conj(), target_sv)) ** 2
    return -fid


# Optimise for even state
print("\nOptimising even-parity circuit...")
rng = np.random.default_rng(42)
best_even = None
for _trial in range(5):
    x0 = rng.uniform(-np.pi, np.pi, n_params)
    res = minimize(vqe_cost, x0, args=(sv_even,), method="COBYLA", options={"maxiter": 500})
    if best_even is None or res.fun < best_even.fun:
        best_even = res
fid_even = -best_even.fun
print(f"Even fidelity: {fid_even:.6f}")

# Optimise for odd state
print("Optimising odd-parity circuit...")
best_odd = None
for _trial in range(5):
    x0 = rng.uniform(-np.pi, np.pi, n_params)
    res = minimize(vqe_cost, x0, args=(sv_odd,), method="COBYLA", options={"maxiter": 500})
    if best_odd is None or res.fun < best_odd.fun:
        best_odd = res
fid_odd = -best_odd.fun
print(f"Odd fidelity: {fid_odd:.6f}")

# Build equal-depth circuits
qc_even = ansatz.assign_parameters(best_even.x)
qc_odd = ansatz.assign_parameters(best_odd.x)

# Verify EQUAL depth
print(f"\nEven depth: {qc_even.depth()}")
print(f"Odd depth:  {qc_odd.depth()}")
assert qc_even.depth() == qc_odd.depth(), "DEPTH MISMATCH!"
print("EQUAL DEPTH CONFIRMED")

In [ ]:
# Add noise layers (same for both) and measure
def add_noise_layers(qc, n_layers=5):
    qc_noisy = qc.copy()
    for _ in range(n_layers):
        for i in range(N):
            qc_noisy.rz(0.01, i)
        for i in range(N - 1):
            qc_noisy.cx(i, i + 1)
            qc_noisy.cx(i, i + 1)  # CX² = I (accumulates noise)
    qc_noisy.measure_all()
    return qc_noisy


qc_even_noisy = add_noise_layers(qc_even)
qc_odd_noisy = add_noise_layers(qc_odd)
print(f"Even+noise depth: {qc_even_noisy.depth()}")
print(f"Odd+noise depth:  {qc_odd_noisy.depth()}")


# Simulator baseline with Heron-like noise
def heron_noise(depol=0.01):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(
        thermal_relaxation_error(50.0, 30.0, 0.1), ["rx", "ry", "rz", "sx", "x"]
    )
    nm.add_all_qubit_quantum_error(depolarizing_error(depol, 2), ["cx", "ecr"])
    return nm


def classical_fidelity(ideal_sv, counts, n_qubits):
    probs_ideal = np.abs(ideal_sv) ** 2
    total = sum(counts.values())
    probs_meas = np.zeros(2**n_qubits)
    for bitstr, count in counts.items():
        probs_meas[int(bitstr, 2)] = count / total
    return float(np.sum(np.sqrt(probs_ideal * probs_meas + 1e-20))) ** 2


sim = AerSimulator(noise_model=heron_noise(0.02))
shots = 10000

F_even_sim, F_odd_sim = [], []
for seed in range(20):
    je = sim.run(qc_even_noisy, shots=shots, seed_simulator=seed)
    jo = sim.run(qc_odd_noisy, shots=shots, seed_simulator=seed)
    F_even_sim.append(classical_fidelity(sv_even, je.result().get_counts(), N))
    F_odd_sim.append(classical_fidelity(sv_odd, jo.result().get_counts(), N))

from scipy.stats import ttest_rel

t_sim, p_sim = ttest_rel(F_even_sim, F_odd_sim)

print("\n=== SIMULATOR (equal-depth, Heron noise) ===")
print(f"Even: {np.mean(F_even_sim):.4f} ± {np.std(F_even_sim):.4f}")
print(f"Odd:  {np.mean(F_odd_sim):.4f} ± {np.std(F_odd_sim):.4f}")
print(f"t={t_sim:.3f}, p={p_sim:.6f}")
direction = "ODD > EVEN" if np.mean(F_odd_sim) > np.mean(F_even_sim) else "EVEN > ODD"
print(f"Direction: {direction}")
print(f"Significant: {'YES' if p_sim < 0.05 else 'NO'}")

## Experiment B: Magnetisation Sector Decoherence

NB38 showed FIM creates energy barriers between M sectors.
On hardware, different M sectors may decohere at different rates
even WITHOUT explicit FIM — because the XY Hamiltonian already has
M-dependent structure (sector-resolved energetics).

Prepare computational basis states with known M, apply noise, measure.
M=0: |0101⟩ or |0110⟩ (equal up/down)
M=±2: |0001⟩ or |1110⟩ (one flip from aligned)
M=±4: |0000⟩ or |1111⟩ (fully aligned)

In [ ]:
# Prepare states with known M
sector_states = {
    "M=+4 (|0000⟩)": "0000",
    "M=+2 (|0001⟩)": "0001",
    "M=0  (|0101⟩)": "0101",
    "M=-2 (|0111⟩)": "0111",
    "M=-4 (|1111⟩)": "1111",
}


def prepare_and_noise(bitstring, n_noise_layers=8):
    """Prepare a computational basis state, add noise layers, measure."""
    n = len(bitstring)
    qc = QuantumCircuit(n)
    for i, b in enumerate(bitstring):
        if b == "1":
            qc.x(i)
    # Noise layers (all same depth)
    for _ in range(n_noise_layers):
        for i in range(n):
            qc.rz(0.01, i)
        for i in range(n - 1):
            qc.cx(i, i + 1)
            qc.cx(i, i + 1)
    qc.measure_all()
    return qc


# All circuits have EQUAL depth
sector_circuits = {}
for name, bits in sector_states.items():
    qc = prepare_and_noise(bits)
    sector_circuits[name] = qc
    print(f"{name}: depth={qc.depth()}")

# Simulator test
print("\n=== SIMULATOR: SECTOR DECOHERENCE ===")
print(f"{'Sector':>16} {'Survival':>10} {'Entropy':>10}")

sector_results = {}
for name, bits in sector_states.items():
    qc = sector_circuits[name]
    survivals = []
    for seed in range(20):
        job = sim.run(qc, shots=shots, seed_simulator=seed)
        counts = job.result().get_counts()
        # Survival probability = fraction still in original state
        survival = counts.get(bits, 0) / shots
        survivals.append(survival)
    mean_surv = np.mean(survivals)
    # Shannon entropy of output distribution
    job = sim.run(qc, shots=shots * 10, seed_simulator=999)
    counts = job.result().get_counts()
    probs = np.array(list(counts.values())) / sum(counts.values())
    entropy = -np.sum(probs * np.log2(probs + 1e-10))

    sector_results[name] = {"survival": round(mean_surv, 4), "entropy": round(entropy, 4)}
    print(f"{name:>16} {mean_surv:10.4f} {entropy:10.4f}")

## Experiment C: FIM vs Standard Ground State Robustness

Prepare ground states of H_XY and H_FIM(λ=2) via equal-depth VQE.
Apply identical noise. If FIM ground state is more robust → dual protection.

In [ ]:
# Ground states
gs_xy = V_xy[:, 0]
gs_fim = V_fim[:, 0]

# VQE for both with same ansatz
print("Optimising H_XY ground state circuit...")
best_xy = None
for _trial in range(5):
    x0 = rng.uniform(-np.pi, np.pi, n_params)
    res = minimize(vqe_cost, x0, args=(gs_xy,), method="COBYLA", options={"maxiter": 500})
    if best_xy is None or res.fun < best_xy.fun:
        best_xy = res
print(f"  Fidelity: {-best_xy.fun:.6f}")

print("Optimising H_FIM ground state circuit...")
best_fim = None
for _trial in range(5):
    x0 = rng.uniform(-np.pi, np.pi, n_params)
    res = minimize(vqe_cost, x0, args=(gs_fim,), method="COBYLA", options={"maxiter": 500})
    if best_fim is None or res.fun < best_fim.fun:
        best_fim = res
print(f"  Fidelity: {-best_fim.fun:.6f}")

qc_xy = add_noise_layers(ansatz.assign_parameters(best_xy.x))
qc_fim = add_noise_layers(ansatz.assign_parameters(best_fim.x))
print(f"\nH_XY circuit depth:  {qc_xy.depth()}")
print(f"H_FIM circuit depth: {qc_fim.depth()}")

# Simulator
F_xy, F_fim = [], []
for seed in range(20):
    j1 = sim.run(qc_xy, shots=shots, seed_simulator=seed)
    j2 = sim.run(qc_fim, shots=shots, seed_simulator=seed)
    F_xy.append(classical_fidelity(gs_xy, j1.result().get_counts(), N))
    F_fim.append(classical_fidelity(gs_fim, j2.result().get_counts(), N))

t_c, p_c = ttest_rel(F_xy, F_fim)
print("\n=== SIMULATOR: FIM vs XY GROUND STATE ===")
print(f"H_XY:  F={np.mean(F_xy):.4f} ± {np.std(F_xy):.4f}")
print(f"H_FIM: F={np.mean(F_fim):.4f} ± {np.std(F_fim):.4f}")
print(f"t={t_c:.3f}, p={p_c:.6f}")
if np.mean(F_fim) > np.mean(F_xy):
    print("FIM GROUND STATE MORE ROBUST (dual protection confirmed on simulator)")
else:
    print("XY ground state more robust (dual protection NOT confirmed on simulator)")

In [ ]:
# === IBM HARDWARE SUBMISSION ===
IBM_RUN = False  # Set True when ready

if IBM_RUN:
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

    service = QiskitRuntimeService(
        channel="ibm_cloud",
        token="REDACTED",
        instance="crn:v1:bluemix:public:quantum-computing:us-east:"
        "a/78db885720334fd19191b33a839d0c35:"
        "841cc36d-0afd-4f96-ada2-8c56e1c443a0::",
    )
    backend = service.backend("ibm_fez")
    pm = generate_preset_pass_manager(backend=backend, optimization_level=2)
    sampler = SamplerV2(backend)

    # Transpile all circuits
    circuits_hw = {
        "exp_A_even": pm.run(qc_even_noisy),
        "exp_A_odd": pm.run(qc_odd_noisy),
        "exp_C_xy": pm.run(qc_xy),
        "exp_C_fim": pm.run(qc_fim),
    }
    for name, _bits in sector_states.items():
        circuits_hw[f"exp_B_{name[:4]}"] = pm.run(sector_circuits[name])

    # Verify depths
    for name, qc in circuits_hw.items():
        print(f"{name}: {qc.depth()} depth after transpile")

    # Submit
    print("\nSubmitting all experiments...")
    jobs = {}
    for name, qc in circuits_hw.items():
        job = sampler.run([qc] * 10, shots=8192)
        jobs[name] = job.job_id()
        print(f"  {name}: {job.job_id()}")

    # Save job IDs
    results_dir = Path("results/ibm_hardware_v2_2026-03-29")
    results_dir.mkdir(exist_ok=True)
    with open(results_dir / "job_ids.json", "w") as f:
        json.dump(jobs, f, indent=2)
    print("\nJob IDs saved. Estimated QPU: ~4 min for all experiments.")
else:
    print("IBM_RUN=False. Set True to submit to hardware.")
    print("Estimated budget: ~4 min of 8 min remaining.")

In [ ]:
# Save simulator results
output = {
    "experiment": "IBM Hardware Tests v2 — fair DLA parity + sector decoherence + FIM protection",
    "n_qubits": N,
    "ansatz": f"EfficientSU2 reps={REPS}",
    "exp_A_parity": {
        "circuit_depth": qc_even_noisy.depth(),
        "vqe_fidelity_even": round(fid_even, 6),
        "vqe_fidelity_odd": round(fid_odd, 6),
        "simulator": {
            "F_even": round(float(np.mean(F_even_sim)), 4),
            "F_odd": round(float(np.mean(F_odd_sim)), 4),
            "t_stat": round(float(t_sim), 4),
            "p_value": float(p_sim),
            "direction": direction,
        },
    },
    "exp_B_sectors": sector_results,
    "exp_C_fim_protection": {
        "simulator": {
            "F_xy": round(float(np.mean(F_xy)), 4),
            "F_fim": round(float(np.mean(F_fim)), 4),
            "t_stat": round(float(t_c), 4),
            "p_value": float(p_c),
        },
    },
}
out_path = Path(
    "/media/anulum/724AA8E84AA8AA75/aaa_God_of_the_Math_Collection/03_CODE/scpn-quantum-control/results/ibm_v2_simulator_baseline_2026-03-29.json"
)
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"Simulator baseline saved to {out_path}")

print("\n=== EXPERIMENT DESIGN SUMMARY ===")
print(
    f"A: Equal-depth parity — even depth={qc_even_noisy.depth()}, odd depth={qc_odd_noisy.depth()}"
)
print(f"B: 5 magnetisation sectors — all depth={list(sector_circuits.values())[0].depth()}")
print(f"C: FIM vs XY ground state — both depth={qc_xy.depth()}")
print(f"Total circuits: {2 + 5 + 2} = 9, each × 10 reps × 8192 shots")
print("Estimated QPU time: ~4 min")